# WISER Module 5 — Data Visualization and Communication

Create one accessible chart and a one-page PDF decision brief for a decision-making audience. This notebook supports **WISER Sections 5.3–5.4**.

Please integrate the use of any AI tools you have to complete this colab.

**Choose one pathway:** Complete **Foundational** or **Applied**, matching your choice in WISER. Follow the corresponding prompts and submission checklist below.

**About this activity:** The supplied `post_overdose_data.csv` contains synthetic event records, without verified death counts or population denominators. The guided chart shows recorded events; it cannot establish overdose mortality or population rates. Applied learners adapt a permitted public aggregate source and document a HEAL CDE crosswalk. Complete the ~200-word tradeoff reflection.

**Use permitted data:** Do not enter PHI, identifiable clinical records or images, restricted participant data, or other sensitive information.


## Start here

1. Open this notebook in Google Colab. If needed, choose **File → Upload notebook** and select the `.ipynb` file.
2. Choose **File → Save a copy in Drive**. Work in that copy.
3. Set `PATHWAY` in the next code cell to `"Foundational"` or `"Applied"`, matching WISER.
4. Work from top to bottom. Click **▶** beside a code cell or press **Shift+Enter** to run it. Read its output before moving on.

**How to enter answers:** Edit the code cell, not the table or message below it. Replace the entire `[bracketed prompt]`, including brackets, with your own answer. Keep quotation marks, field names, commas, and braces. For a table, edit the quoted values in each row of the code; then run it to rebuild the displayed table. Code colors are syntax highlighting, not edit instructions.

**Action labels:** **Run** means no edit is required. **Edit, then run** names the fields to change. **Run, then answer** means inspect the output first and complete the response cell that follows. Complete the work for your pathway; leave the other pathway's answer fields unchanged.

**After a pause:** Save first. When you return, rerun code cells from the top, keeping your answers. If you change an earlier answer or setting, rerun that cell and the cells below it so the saved files match.

**If an error appears:** Read the first error below the cell. Correct the named issue and rerun. If you need help, share that error and its section name with your instructor.

### How this module builds your skills

Build on Module 4: use the same habit of checking definitions before making a claim. Now choose an audience, describe a chart accurately, and recommend a proportionate action. The plotting code is supplied; your interpretation is more independent.

### Your Module 5 checklist

| Stage | What you personally do |
| --- | --- |
| Source | Foundational: run the synthetic-data loader. Applied: set `APPLIED_SOURCE` to an appropriate public aggregate CSV and adapt the analysis for its columns; using `"synthetic"` is only offline practice unless allowed by your instructor. |
| Definitions and evidence | Inspect the event fields, document CDE evidence where verified, explain why the provided event file cannot yield a mortality/population rate, and write one equity limitation. |
| Chart and brief | Choose an audience, question and action; write chart title, caption and alt text, then run the plot. Fill the one-page brief's finding, 2–3 caveats and recommendation using values actually shown in the chart. |
| Your pathway | Applied: implement your own source cleaning/denominator/suppression logic and complete the HEAL CDE crosswalk. All learners write a **~200-word** tradeoff reflection. |
| Finish | Run export; open the one-page PDF and outputs ZIP; submit your viewable notebook link, PDF and outputs as specified in WISER. |


> **Next code cell:** **Edit, then run:** Set `PATHWAY` to your WISER choice; keep the capitalization shown.


In [ ]:
PATHWAY = "Foundational"  # Change to "Applied" only if you chose 5.32 in WISER.
assert PATHWAY in ("Foundational", "Applied")
print("Your pathway:", PATHWAY)

## 1. Load and examine your data (WISER 5.4, step 1)

Foundational uses the supplied synthetic event file. Applied uses a documented public aggregate source and adapts the later analysis to its structure.


**Next code cell — **Foundational — run unchanged. Applied — edit, then run:** To use a public CSV, upload it with Colab’s left **Files** panel, copy its path, and paste that path inside the quotes for `APPLIED_SOURCE`. Choose a source suitable for the intended monthly analysis. Use `"synthetic"` only for practice. Check the printed source, first five rows, and missing-value table before continuing.


In [ ]:
from pathlib import Path
import json, re, shutil, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
try:
    from IPython.display import display
except ImportError:
    display = print

DATA_URL = "https://raw.githubusercontent.com/mcbacloudlab/WISERTraining-Modules-/main/Module%205/Data/post_overdose_data.csv"
OUT = Path("module5_outputs")
OUT.mkdir(exist_ok=True)
APPLIED_SOURCE = ""  # Applied: local path to an approved public aggregate CSV, or "synthetic" for offline development.
if PATHWAY == "Foundational" or APPLIED_SOURCE == "synthetic":
    events = pd.read_csv(DATA_URL)
    source_name = "WISER synthetic post_overdose_data.csv"
    source_url = DATA_URL
    synthetic_used = True
elif APPLIED_SOURCE:
    events = pd.read_csv(APPLIED_SOURCE)
    source_name = APPLIED_SOURCE
    source_url = APPLIED_SOURCE
    synthetic_used = False
else:
    raise ValueError("Applied pathway: set APPLIED_SOURCE to a public aggregate CSV path, or 'synthetic' for offline development.")
print("Source:", source_name, "| rows:", len(events), "| columns:", len(events.columns))
display(events.head(5))
display(pd.DataFrame({"field": events.columns, "missing": events.isna().sum().to_numpy()}))

### Read the dataset before charting

One row in the supplied file is an event record, not necessarily a death. `provisional_flag` marks records subject to revision. Counts need a matching population denominator before they can become population rates.


**Next code cell — **Foundational — run:** Check that duplicate IDs and unparseable dates are both zero; record the displayed date range for your caption. A nonzero count stops the cell for review. **Applied:** Add a code cell here to inspect your source’s keys, dates, missingness, inclusion rules, and units; do not assume a public aggregate row is one event.


In [ ]:
if synthetic_used:
    required = {"event_id", "event_date", "opioid_involved", "provisional_flag", "county", "svi_quartile"}
    missing = required - set(events.columns)
    if missing: raise ValueError(f"Required fields missing: {sorted(missing)}")
    events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")
    print("Duplicate event IDs:", events["event_id"].duplicated().sum())
    print("Unparseable dates:", events["event_date"].isna().sum())
    print("Period:", events["event_date"].min().date(), "to", events["event_date"].max().date())
    print("Provisional rows by year:")
    display(pd.crosstab(events["event_date"].dt.year, events["provisional_flag"]))
    if events["event_id"].duplicated().any() or events["event_date"].isna().any():
        raise ValueError("Review duplicate IDs or dates before plotting.")
else:
    print("APPLIED TASK: document source structure, duplicates, missingness, inclusion criteria, and cleaning in your own cells.")

## 2. Record definitions and CDE alignment (WISER 5.4, step 2)

A candidate CDE topic is not a verified mapping. Compare definitions and record evidence before entering an identifier, as in Module 4.


**Next code cell — **Foundational — run and read:** Review the source definitions and candidate CDE topics; the supplied `unverified` entries may remain. **Applied — edit, then run:** Replace the placeholder public-source table with one dictionary per actual source field. Use the definition-checking method from Module 4: compare source and target wording, timeframe, and units; record a reference in `evidence`. Enter an ID only if verified; otherwise document `unverified` or `no suitable mapping` and why. Check that the displayed rows describe your source.


In [ ]:
cde_crosswalk = pd.DataFrame([
    {"source_field":"event_date", "source_definition":"Date of synthetic event", "candidate_CDE_topic":"time", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"county", "source_definition":"County assigned to synthetic event", "candidate_CDE_topic":"geography", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"substance_involved_primary", "source_definition":"Primary substance label, not necessarily all substances involved", "candidate_CDE_topic":"drug class", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"age_group", "source_definition":"Age category", "candidate_CDE_topic":"demographics", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
    {"source_field":"provisional_flag", "source_definition":"Event record is flagged provisional", "candidate_CDE_topic":"data status", "verified_target_id":"", "mapping_status":"unverified", "evidence":""},
])
if PATHWAY == "Applied" and not synthetic_used:
    # Replace with documented fields from your actual public source.
    cde_crosswalk = pd.DataFrame([{col: "" for col in cde_crosswalk.columns}])
    print("APPLIED: populate with your source fields and verified HEAL CDE references.")
display(cde_crosswalk)

## 3. Clean, count, and assess rate readiness (WISER 5.4, steps 3–4)

The supplied analysis counts recorded opioid-involved synthetic events by month. It lacks both a verified death outcome and a matching population denominator, so it cannot establish a mortality rate. Public-source analyses must document outcome definitions, reporting coverage, and any denominator/suppression rules.


**Next code cell — **Foundational — run:** Inspect `monthly`; one row represents one month. Check `recorded_events`, `provisional_events`, and `provisional`. Read why a population or mortality rate cannot be calculated. **Applied:** Before this cell, add your cleaning code so `events` has one row per month and columns `month` (date), `recorded_events` (numeric count), and `provisional` (Boolean `True`/`False`). Document aggregation and suppression handling. If your source cannot support monthly counts, adapt the analysis and plotting cells consistently rather than inventing months or counts.


In [ ]:
if synthetic_used:
    analysis = events.loc[events["opioid_involved"].eq(1)].copy()
    analysis["month"] = analysis["event_date"].dt.to_period("M").dt.to_timestamp()
    monthly = (analysis.groupby("month", as_index=False)
              .agg(recorded_events=("event_id", "size"), provisional_events=("provisional_flag", "sum")))
    all_months = pd.date_range(events["event_date"].min().to_period("M").to_timestamp(),
                               events["event_date"].max().to_period("M").to_timestamp(), freq="MS")
    monthly = pd.DataFrame({"month":all_months}).merge(monthly, on="month", how="left")
    monthly[["recorded_events", "provisional_events"]] = monthly[["recorded_events", "provisional_events"]].fillna(0).astype(int)
    # A month is marked provisional when at least one of its source records is flagged.
    monthly["provisional"] = monthly["provisional_events"].gt(0)
    print("Outcome: count of recorded opioid-involved synthetic events per month")
    print("Population rate available? No; matching population denominator is absent.")
    print("Overdose mortality rate available? No; no verified fatal outcome or denominator.")
    display(monthly.tail(15))
else:
    # Applied learners implement and document the transformation from their public
    # aggregate source to a plot-ready table; this accepted schema is an example.
    needed = {"month", "recorded_events", "provisional"}
    if not needed.issubset(events.columns):
        raise ValueError("APPLIED: add a cleaning cell that creates month, recorded_events, and provisional columns; document any population rate separately. Found columns: " + str(list(events.columns)))
    monthly = events[["month", "recorded_events", "provisional"]].copy()
    monthly["month"] = pd.to_datetime(monthly["month"], errors="coerce")
    monthly["recorded_events"] = pd.to_numeric(monthly["recorded_events"], errors="coerce")
    if monthly["month"].isna().any() or monthly["recorded_events"].isna().any():
        raise ValueError("APPLIED: correct missing/non-numeric dates or counts in your public table.")
    if monthly["month"].duplicated().any():
        raise ValueError("APPLIED: aggregate to exactly one row per month and document that aggregation.")
    monthly["provisional"] = monthly["provisional"].astype(bool)
    monthly = monthly.sort_values("month").reset_index(drop=True)
    print("APPLIED: verify that labels, measure, periods, provenance, and CDEs below describe your own data.")

## 4. Show uncertainty and consider equity (WISER 5.4, steps 5–6)

The chart will mark months based on the **actual** `provisional_flag`, rather than guessing that the last six months are provisional. This flag identifies records subject to revision; it does not tell us the size or direction of a future revision. A month with zero events may also reflect reporting coverage, so avoid claiming the population had no events.

The file includes `svi_quartile`, but it lacks tract-level denominators and a documented SVI merge. Examine it for context; do not rank communities or infer risk from raw counts. Write one equity limitation.


**Next code cell — **Run, then answer:** Inspect the SVI table if using the synthetic file. In `equity_note`, write 1–2 sentences naming information missing from a fair comparison, such as population denominators or reporting coverage. Rerun the cell and check your answer prints. Applied: discuss the corresponding limitation of your own source.


In [ ]:
if synthetic_used:
    display(events.groupby("svi_quartile", as_index=False).agg(recorded_events=("event_id", "size"), opioid_involved=("opioid_involved", "sum")))
    print("This is a descriptive count in a synthetic sample, not a community rate.")
equity_note = """[Describe one limitation of comparing these event counts by SVI or county without a population denominator or coverage information.]"""
print(equity_note)

## 5. Plan an accessible chart (WISER 5.1 and 5.4, step 7)

A line chart suits a question about change over time. Labels, caption, and alt text should describe the measure actually plotted; provisional status and source limitations belong in the explanation.


**Next code cell — **Edit, then run:** Set `audience`, `question`, and `action_goal` for one decision-maker. Use the monthly table to set `chart_title` and `chart_caption`; include the measure and actual date range. Draft `alt_text` from the monthly values, then refine it after viewing the chart in the next cell. Keep the count label for the supplied data. The accessibility response comes in the later language-audit cell, not here.


In [ ]:
audience = "County health department planning team"  # EDIT: choose your decision-maker.
question = "How did recorded opioid-involved synthetic event counts vary by month?"  # EDIT for your chosen audience.
action_goal = "Review outreach capacity and data quality before allocating resources."  # EDIT to a specific defensible action.
chart_title = "Recorded opioid-involved events by month"  # EDIT: say what is measured.
chart_caption = "Synthetic WISER events, Jan 2019–Mar 2025. Monthly counts; shaded months contain provisional records. Not a mortality or population rate."  # EDIT.
alt_text = """[Describe the high/low pattern or change visible in the chart, including the provisional period. Do not just write 'line chart'.]"""
print("Audience:", audience)
print("Question:", question)
print("Decision:", action_goal)

> **Next code cell:** **Run:** Draw the chart and check that its dates and provisional shading match the data. If you changed earlier chart fields, run this again.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(monthly["month"], monthly["recorded_events"], color="#12647A", linewidth=2.3, marker="o", markersize=2.3)
# Shade each contiguous run of months containing provisional records.
for row in monthly.itertuples():
    if row.provisional:
        ax.axvspan(row.month, row.month + pd.DateOffset(months=1), color="#F1C04A", alpha=.34)
ax.set(title=chart_title, xlabel="Month", ylabel="Recorded events (count)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#DFE5E8")
ax.spines[["top", "right"]].set_visible(False)
ax.text(.99, .96, "Gold shading: provisional records", transform=ax.transAxes,
        ha="right", va="top", fontsize=9, bbox={"facecolor":"white","alpha":.9,"edgecolor":"none"})
fig.autofmt_xdate(rotation=0)
fig.tight_layout()
chart_path = OUT / "module5_primary_chart.png"
fig.savefig(chart_path, dpi=180, bbox_inches="tight")
plt.show()
print("Chart saved:", chart_path)
print("Alt text:", alt_text)

### Chart and language audit

The word check flags wording for human review; it cannot decide appropriateness in context. Inspect both the labels and the chart.


**Next code cell — **Run, then answer:** Read the wording flags and inspect the chart yourself. If needed, edit the earlier chart-planning fields and rerun from there. Replace the bracketed `accessibility_check` in this cell with the checks you made and one revision (or explain why none was needed). Rerun this cell to save that response.


In [ ]:
TERMS_TO_REVIEW = ["addict", "abuser", "drug abuse", "substance abuse", "junkie", "dirty", "clean"]
def wording_flags(text):
    return [term for term in TERMS_TO_REVIEW if re.search(r"(?<!\w)"+re.escape(term)+r"(?!\w)", str(text), re.I)]
for label, value in {"title":chart_title, "caption":chart_caption, "alt text":alt_text, "question":question}.items():
    print(label, "terms to review:", wording_flags(value))
accessibility_check = """[Confirm the unit, date range, source/caption, provisional cue, readable labels, alt text, and person-first wording. Note one change you made.]"""
print(accessibility_check)

## 6. Draft the one-page decision brief (WISER 5.1 and 5.4, step 8)

The brief combines your question, finding, chart, caveats, and recommended action for one audience. A supported finding and a proportionate next step are more useful than an unsupported population claim.


**Next code cell — **Edit, then run:** Use the chart to write `headline`; include a value you can locate in `monthly`. Fill the first two `caveats`; fill the third or set it to `""` if unused. In `recommendation`, name an action, owner, and timeframe. Check `source_note` against the loaded source and dates, updating it for Applied data. Keep headline/action within 170 characters and each caveat within 110 characters for the PDF.


In [ ]:
headline = "[Write one plain-language finding from the actual chart, with a number if you can support it.]"
caveats = [
    "[Describe the provisional months and how they limit interpretation.]",
    "[Explain that these are synthetic recorded events, not mortality rates or population rates.]",
    "[Optional: note reporting coverage, missing denominator, or equity limitation.]",
]
recommendation = "[State a specific, proportionate planning or data-quality action, who should take it, and when.]"
source_note = "WISER synthetic post_overdose_data.csv; selected opioid-involved records. Data through March 2025."
print("Question:", question, "\nFinding:", headline, "\nCaveats:", *caveats, sep="\n")
print("Action:", recommendation)

> **Next code cell:** **Run:** Export the one-page PDF; if a text-length error appears, shorten the named field in the prior cell and rerun.


In [ ]:
# Keep the brief within one readable page. Shorten fields rather than allowing overlap.
limits = {"audience":100, "question":170, "headline":170, "caption":190,
          "recommendation":170, **{f"caveat {i}":110 for i in range(1,4)}}
values = {"audience":audience, "question":question, "headline":headline,
          "caption":chart_caption, "recommendation":recommendation,
          **{f"caveat {i}":c for i,c in enumerate(caveats,1)}}
long = [k for k,v in values.items() if len(v)>limits[k]]
if long: raise ValueError("Shorten these fields to fit the one-page brief: " + ", ".join(long))

brief_fig = plt.figure(figsize=(8.5, 11), facecolor="white")
brief_fig.text(.075, .966, "WISER Module 5 | Decision brief", fontsize=14, weight="bold", color="#143C4A")
brief_fig.text(.075, .935, f"Audience: {audience}", fontsize=9)
brief_fig.text(.075, .905, "Question", fontsize=10, weight="bold")
brief_fig.text(.075, .884, "\n".join(textwrap.wrap(question, 95)), fontsize=9.2, va="top")
brief_fig.text(.075, .825, "Key finding", fontsize=10, weight="bold")
brief_fig.text(.075, .804, "\n".join(textwrap.wrap(headline, 95)), fontsize=9.2, va="top")
chart_ax = brief_fig.add_axes([.075, .405, .85, .34])
chart_ax.imshow(plt.imread(chart_path))
chart_ax.axis("off")
brief_fig.text(.075, .394, "Figure. " + "\n".join(textwrap.wrap(chart_caption, 105)), fontsize=8.3, va="top")
brief_fig.text(.075, .33, "Key caveats", fontsize=10, weight="bold")
y = .306
for i, caveat in enumerate(caveats, 1):
    if caveat.strip():
        lines = textwrap.wrap(f"{i}. {caveat}", 112)
        brief_fig.text(.075, y, "\n".join(lines), fontsize=8.8, va="top", linespacing=1.2)
        y -= .018*len(lines) + .012
if y < .145: raise ValueError("Shorten the caveats to keep the brief on one readable page.")
brief_fig.text(.075, .141, "Recommended action", fontsize=10, weight="bold")
brief_fig.text(.075, .119, "\n".join(textwrap.wrap(recommendation, 105)), fontsize=8.8, va="top")
brief_fig.text(.075, .042, "Source: " + source_note, fontsize=7.5)
brief_fig.text(.075, .025, "Learning exercise | Synthetic event records; no population or mortality estimate" if synthetic_used else "Public data | See notebook and metadata for methods", fontsize=7.5)
brief_path = OUT / "Module_5_one_page_decision_brief.pdf"
with PdfPages(brief_path) as pdf:
    pdf.savefig(brief_fig)
plt.close(brief_fig)
print("Saved one-page PDF:", brief_path)
print("Review the PDF visually before submitting; edit text and rerun this cell if anything overlaps.")

## 7. Applied pathway extension (WISER 5.32)

Applied extends the guided workflow to public data: ingestion, cleaning, appropriate aggregation, denominator/suppression decisions, charting, and a documented HEAL CDE crosswalk. An optional SUDORS or SVI merge requires compatible geography, periods, definitions, and access terms. The synthetic file supports disclosed practice rather than a final public-data analysis.


**Next code cell — **Foundational — run unchanged and continue. Applied — document your completed extension:** The public-data import and cleaning code belongs before the analysis cells that use it, not at the end. Return to Sections 1–3 to implement it, then rerun downward. Complete `applied_methods` here with the source/release, definitions, geography, denominator availability, suppression rules, and cleaning decisions. Check that the chart, brief, and CDE crosswalk all describe that source.


In [ ]:
applied_methods = """[Applied: describe source URL/release, public-use status, outcome and drug codes, geography, denominator, suppression, cleaning, and revisions. Add your implementation code cells above.]"""
if PATHWAY == "Applied":
    print("APPLIED CHECK: a public-data pipeline and verified CDE mapping are required; this text is only a prompt.")
    print(applied_methods)

## 8. Reflect, check, and export (WISER 5.3–5.4, step 9)

Write **about 200 words** on one tradeoff (for example: count versus rate, including provisional months, or omitting a county comparison). Explain what you chose, why, and what information would change your choice. Run the last cell after finishing the chart, brief, and responses.

**Submit:** (1) Colab notebook link with view access; (2) one-page PDF brief; (3) outputs folder/ZIP. Your reflection is saved in the notebook and outputs. For Applied, include the verified HEAL CDE crosswalk. Open the PDF and files before you submit.


> **Next code cell:** **Edit, then run:** Replace `tradeoff_reflection` with your ~200-word explanation of one chart or evidence choice.


In [ ]:
tradeoff_reflection = """[Replace with your ~200-word reflection on one evidence or communication tradeoff.]"""
print("Reflection words:", len(tradeoff_reflection.split()))

> **Next code cell:** **Run last:** Resolve any `TO COMPLETE` messages, inspect the generated files, and download the ZIP.


In [ ]:
required_text = {"alt text":alt_text, "accessibility check":accessibility_check,
                 "finding":headline, "action":recommendation, "tradeoff reflection":tradeoff_reflection,
                 "equity note":equity_note}
for i, caveat in enumerate(caveats, 1): required_text[f"caveat {i}"] = caveat
unfinished = [key for key,value in required_text.items() if "[" in str(value) and "]" in str(value)]
if unfinished: print("TO COMPLETE before submission:", ", ".join(unfinished))
if not 170 <= len(tradeoff_reflection.split()) <= 230:
    print("TO COMPLETE: write a roughly 200-word tradeoff reflection.")
if PATHWAY == "Applied" and (synthetic_used or "[" in applied_methods or cde_crosswalk["mapping_status"].eq("unverified").all()):
    print("APPLIED CHECK: complete your public-data pipeline and documented CDE crosswalk; disclose any synthetic fallback.")

monthly.to_csv(OUT / "monthly_recorded_events.csv", index=False)
cde_crosswalk.to_csv(OUT / "cde_alignment_table.csv", index=False)
(OUT / "reflection.txt").write_text(tradeoff_reflection.strip()+"\n", encoding="utf-8")
(OUT / "chart_alt_text.txt").write_text(alt_text.strip()+"\n", encoding="utf-8")
(OUT / "brief_text.md").write_text(
    f"# Decision brief\n\nAudience: {audience}\n\nQuestion: {question}\n\n"
    f"Finding: {headline}\n\nCaveats:\n"+"\n".join(f"- {x}" for x in caveats)
    +f"\n\nRecommended action: {recommendation}\n\nSource: {source_note}\n", encoding="utf-8")
metadata = {"module":"WISER Module 5", "pathway":PATHWAY, "source":source_name,
            "source_url_or_path":source_url, "rows":len(events), "synthetic_used":synthetic_used,
            "measure":"monthly count of recorded opioid-involved synthetic events" if synthetic_used else "[Applied: document actual measure]",
            "population_denominator_available":False if synthetic_used else "[verify]",
            "fatal_outcome_available":False if synthetic_used else "[verify]",
            "provisional_definition":"month has one or more flagged source records" if synthetic_used else "[verify]",
            "figure_file":chart_path.name, "brief_file":brief_path.name}
(OUT / "metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
(OUT / "README.md").write_text(
    f"# WISER Module 5 outputs\n\nPathway: {PATHWAY}\nSource: {source_name}\n\n"
    "Files: primary chart (PNG), one-page brief (PDF), monthly_recorded_events.csv, "
    "cde_alignment_table.csv, brief_text.md, chart_alt_text.txt, reflection.txt, metadata.json.\n\n"
    "For the supplied synthetic file, the measure is recorded opioid-involved event count, "
    "not a mortality count or a population rate. Provisional months are flagged from source rows. "
    "The CDE table contains topic-level candidates until target IDs and definitions are verified.\n",
    encoding="utf-8")
archive = shutil.make_archive("WISER_Module_5_outputs", "zip", root_dir=".", base_dir=str(OUT))
print("PDF:", brief_path, "| ZIP:", archive)
print("Files:", sorted(x.name for x in OUT.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Outside Colab: download the ZIP from the file browser.")